In [ ]:
sentences = [
    "the cat is sleeping",
    "the dog is sleeping",
    "the cat is running",
    "the dog is running",
    "the cat is eating",
    "the dog is eating"
]

In [2]:
train_words= [word for sentence in sentences for word in sentence]

In [5]:
print(len(train_words))


108


In [6]:
print(train_words)

['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 'i', 's', ' ', 's', 'l', 'e', 'e', 'p', 'i', 'n', 'g', 't', 'h', 'e', ' ', 'd', 'o', 'g', ' ', 'i', 's', ' ', 's', 'l', 'e', 'e', 'p', 'i', 'n', 'g', 't', 'h', 'e', ' ', 'c', 'a', 't', ' ', 'i', 's', ' ', 'r', 'u', 'n', 'n', 'i', 'n', 'g', 't', 'h', 'e', ' ', 'd', 'o', 'g', ' ', 'i', 's', ' ', 'r', 'u', 'n', 'n', 'i', 'n', 'g', 't', 'h', 'e', ' ', 'c', 'a', 't', ' ', 'i', 's', ' ', 'e', 'a', 't', 'i', 'n', 'g', 't', 'h', 'e', ' ', 'd', 'o', 'g', ' ', 'i', 's', ' ', 'e', 'a', 't', 'i', 'n', 'g']


In [7]:
import nltk


In [8]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
from nltk.tokenize import word_tokenize



In [11]:
sentences_tokens= [word_tokenize(sentence) for sentence in sentences]

In [12]:
print(sentences_tokens)

[['the', 'cat', 'is', 'sleeping'], ['the', 'dog', 'is', 'sleeping'], ['the', 'cat', 'is', 'running'], ['the', 'dog', 'is', 'running'], ['the', 'cat', 'is', 'eating'], ['the', 'dog', 'is', 'eating']]


In [13]:
tokens=[]
for sentence in sentences:
  tokens.append(word_tokenize(sentence))

In [14]:
print(tokens)

[['the', 'cat', 'is', 'sleeping'], ['the', 'dog', 'is', 'sleeping'], ['the', 'cat', 'is', 'running'], ['the', 'dog', 'is', 'running'], ['the', 'cat', 'is', 'eating'], ['the', 'dog', 'is', 'eating']]


In [15]:
print(len(tokens))

6


In [16]:
token_words=[word for sentence in tokens for word in sentence]

In [17]:
print(len(token_words))

24


In [18]:
from collections import Counter
counter = Counter(token_words)
word_to_id={}
for word in counter:
  if word not in word_to_id:
    word_to_id[word]=len(word_to_id)


In [19]:
id_to_word={idx:word for word, idx in word_to_id.items()}

In [21]:
vocab_size=len(word_to_id)
print(vocab_size)

7


In [24]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
model=Sequential([
    Embedding(vocab_size,32),
    LSTM(64),
    Dense(vocab_size, activation="softmax")
])

In [25]:
sequence_length = max(len(sentence) for sentence in sentences_tokens)

In [26]:
import tensorflow as tf
from tensorflow import keras

sequence_length = 3

inputs = keras.Input(shape=(sequence_length,))

x = keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=32
)(inputs)

attention = keras.layers.MultiHeadAttention(
    num_heads=2,
    key_dim=32
)(x, x)

x = keras.layers.Add()([x, attention])

x = keras.layers.LayerNormalization()(x)

x = keras.layers.GlobalAveragePooling1D()(x)

outputs = keras.layers.Dense(
    vocab_size,
    activation="softmax"
)(x)

transformer_model = keras.Model(
    inputs=inputs,
    outputs=outputs
)

transformer_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 3, 32)     │        224 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 3, 32)     │      8,416 │ embedding_1[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 3, 32)     │          0 │ embedding_1[0][0… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 3, 32)     │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 7)         │        231 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,935 (34.90 KB)

 Trainable params: 8,935 (34.90 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [1]:
from google.colab import files
uploads=files.upload()